[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/09_Root_Finding.ipynb)

# Root Finding

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 7

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

## Learning goals

By the end of this notebook you should be able to:

- recognize when an equation must be solved numerically, and rewrite it in the form $f(x) = 0$;
- locate roots graphically, then refine them with **bisection**, **Newton–Raphson**, and the **secant method**;
- explain how fast each method converges, when each is guaranteed to work, and how Newton's method can fail;
- use `scipy.optimize.brentq`, `newton`, `root_scalar`, and (briefly) `fsolve`;
- find *all* the roots in an interval by scanning for sign changes, and avoid fake "roots" at poles;
- apply root finding to physics: Wien's displacement law, the bound states of a finite square well, and the equilibrium separation of a molecule (as a minimization).

This notebook connects to Project 2, where you will combine root finding with ODE solving to find quantum energy levels.

## 0. Setup

`scipy.optimize` is SciPy's submodule for root finding and minimization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as const
from scipy import optimize

## 1. Why root finding?

Many equations in physics cannot be solved with algebra. A **root** of a function $f$ is a value $x^*$ with $f(x^*) = 0$, and any equation can be written that way: move everything to one side.

**Physics recap: Wien's displacement law.** A hot object (a "blackbody") glows with a spectrum given by Planck's law. As a function of wavelength $\lambda$, the intensity is

$$
I(\lambda, T) \propto \frac{1}{\lambda^5}\,\frac{1}{e^{hc/\lambda k_B T} - 1}.
$$

Hotter objects glow bluer: the peak wavelength $\lambda_\text{max}$ shifts to shorter wavelength as $T$ rises. Wien's law says $\lambda_\text{max} T = b$, a constant. To find $b$, set $dI/d\lambda = 0$. With $x = hc/(\lambda k_B T)$, a few lines of calculus turn this into

$$
5\left(1 - e^{-x}\right) = x ,
$$

and then $b = \dfrac{hc}{k_B\,x}$. There is no way to solve this for $x$ with algebra: $x$ appears both inside and outside an exponential. Equations like this are called **transcendental**. We need

$$
f(x) = 5\left(1 - e^{-x}\right) - x = 0 .
$$

In [ ]:
def f_wien(x):
    return 5 * (1 - np.exp(-x)) - x

## 2. Graphical root finding

**Always plot the function first.** A plot tells you how many roots there are and roughly where, which is exactly what the numerical methods need as a starting point.

In [ ]:
x = np.linspace(-0.5, 8, 400)

fig, ax = plt.subplots()
ax.plot(x, f_wien(x))
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("$x = hc / \\lambda k_B T$")
ax.set_ylabel("$f(x) = 5(1 - e^{-x}) - x$")
ax.grid(alpha=0.3)
plt.show()

There are two roots. $x = 0$ is a trivial root (it corresponds to $\lambda \to \infty$, not the peak). The physical root is near $x \approx 5$. Zooming in with a second plot could give us two or three digits, but that is slow and not precise. The rest of this notebook is about doing better automatically.

## 3. Bisection

**Idea.** If $f$ is continuous and $f(a)$ and $f(b)$ have **opposite signs**, then $f$ must cross zero somewhere between $a$ and $b$ (the intermediate value theorem). The interval $[a, b]$ is said to **bracket** the root. Now:

1. Compute the midpoint $m = (a + b)/2$.
2. If $f(a)$ and $f(m)$ have opposite signs, the root is in $[a, m]$: set $b = m$. Otherwise it is in $[m, b]$: set $a = m$.
3. Repeat until the interval is smaller than a chosen **tolerance**.

Each step halves the interval. A neat trick for "opposite signs": $f(a)\,f(m) < 0$.

We use a `while` loop, which keeps going as long as its condition is `True`. We also record every midpoint in a list `history` so we can study the convergence later; `history.append(m)` adds `m` to the end of the list.

In [ ]:
def bisection(f, a, b, tol=1e-12):
    """Find a root of f in [a, b] by bisection. Returns (root, list of midpoints)."""
    if f(a) * f(b) > 0:
        raise ValueError("f(a) and f(b) must have opposite signs")
    history = []
    while (b - a) / 2 > tol:
        m = (a + b) / 2
        history.append(m)
        if f(a) * f(m) < 0:
            b = m
        else:
            a = m
    return (a + b) / 2, history

In [ ]:
x_bis, hist_bis = bisection(f_wien, 1, 10)
print(f"root = {x_bis:.12f}")
print(f"f(root) = {f_wien(x_bis):.1e}")
print(f"iterations: {len(hist_bis)}")
print("first few midpoints:", np.round(hist_bis[:6], 4))

**How many iterations?** After $n$ steps the interval has width $(b - a)/2^n$. To get below a tolerance $\delta$, we need

$$
n \approx \log_2\frac{b - a}{\delta}.
$$

For $b - a = 9$ and $\delta = 10^{-12}$ that is $\log_2(9\times10^{12}) \approx 43$, matching the count above. Each iteration gains about one *binary* digit, so about 3.3 iterations per decimal digit. This is called **linear convergence**: the error shrinks by a constant factor (here 2) per step.

**Strengths and weaknesses of bisection:**

- It is **guaranteed** to converge, as long as you start with a valid bracket and $f$ is continuous.
- It is slow compared to the methods below.
- It needs a sign change. It cannot find a root where $f$ touches zero without crossing, like $f(x) = x^2$.
- If $f$ jumps from $+\infty$ to $-\infty$ (like $\tan x$ at $\pi/2$), bisection will happily "converge" to the jump, which is not a root. We will meet this problem in the square-well example.
- If the bracket contains several roots, you get one of them, and you don't control which.

If the bracket is invalid, our function raises an error:

In [ ]:
try:
    bisection(f_wien, 6, 10)
except ValueError as err:
    print("ValueError:", err)

### Your turn 1

The equation $\cos x = x$ has one solution. Rewrite it as $f(x) = 0$, plot $f$ to find a bracket, and solve it with `bisection`. Then run bisection with tolerances `1e-3`, `1e-6`, `1e-9`, `1e-12` and print the number of iterations each time. Does the count increase by about the amount $\log_2$ predicts each time you ask for 3 more digits?

In [ ]:
# Your code here

## 4. Newton–Raphson

**Idea.** Near a guess $x_n$, replace $f$ by its tangent line (the first two terms of a Taylor series):

$$
f(x) \approx f(x_n) + f'(x_n)\,(x - x_n).
$$

The tangent line crosses zero at

$$
\boxed{x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}}.
$$

Use $x_{n+1}$ as the next guess and repeat. Newton's method needs a **starting guess** $x_0$ (not a bracket) and the **derivative** $f'$. For Wien's function, $f'(x) = 5e^{-x} - 1$.

We stop when the step $|x_{n+1} - x_n|$ is smaller than the tolerance. We also set a maximum number of iterations, so the loop cannot run forever if the method fails. A `for` loop with `break` does both: `break` exits the loop early.

In [ ]:
def newton(f, fprime, x0, tol=1e-12, max_iter=50):
    """Newton-Raphson iteration. Returns (root, list of iterates)."""
    x = x0
    history = [x]
    for i in range(max_iter):
        x_new = x - f(x) / fprime(x)
        history.append(x_new)
        if abs(x_new - x) < tol:
            break
        x = x_new
    return x_new, history

def fprime_wien(x):
    return 5 * np.exp(-x) - 1

In [ ]:
x_newt, hist_newt = newton(f_wien, fprime_wien, x0=5.0)
x_true = x_newt
for n, xn in enumerate(hist_newt):
    print(f"n = {n}:  x = {xn:.16f}   error = {abs(xn - x_true):.1e}")

`enumerate(list)` gives both the index `n` and the item `xn` in each pass through the loop.

Look at the errors: roughly $10^{-2} \to 10^{-5} \to 10^{-11} \to 0$. The **number of correct digits roughly doubles each step**. This is **quadratic convergence**: $|e_{n+1}| \approx C\,|e_n|^2$. (Taylor-expanding one more order shows $C = |f''/2f'|$ at the root.) Four iterations of Newton beat 43 iterations of bisection.

### 4.1 How Newton's method fails

Newton's method is fast but **not guaranteed**. Classic failure modes:

1. **It finds a different root from the one you wanted.** Start Wien's function at $x_0 = 0.5$, closer to the trivial root:

In [ ]:
root, hist = newton(f_wien, fprime_wien, x0=0.5)
print(f"started at 0.5, converged to {root:.6f}")

2. **The derivative is zero (or tiny).** The tangent line is flat and the next step shoots off to a huge value (or `inf`). For Wien's function $f'(x) = 0$ at $x = \ln 5 \approx 1.609$:

In [ ]:
root, hist = newton(f_wien, fprime_wien, x0=1.6)
print("first iterates:", np.round(hist[:4], 3))
print(f"after {len(hist) - 1} iterations: x = {root:.3f},  f(x) = {f_wien(root):.2e}")

The first step threw it to $x \approx -250$, where $e^{-x}$ is astronomically large and the tangent lines only move it one unit per step. After the maximum 50 iterations it is nowhere near a root, yet our `newton` function returned a number without complaint. **Always check that $f(\text{answer}) \approx 0$.**

3. **It cycles forever.** For $f(x) = x^3 - 2x + 2$ starting at $x_0 = 0$, Newton jumps $0 \to 1 \to 0 \to 1 \to \cdots$.

In [ ]:
g = lambda x: x**3 - 2 * x + 2          # lambda: a one-line function
gprime = lambda x: 3 * x**2 - 2
root, hist = newton(g, gprime, x0=0.0, max_iter=8)
print("iterates:", hist)

(`lambda x: expression` makes a small function in one line, as in the previous notebook.) Without `max_iter`, this would never stop.

4. **It diverges.** For $f(x) = \arctan x$ (root at $0$) starting at $x_0 = 1.5$, the tangent lines throw each guess farther away:

In [ ]:
root, hist = newton(np.arctan, lambda x: 1 / (1 + x**2), x0=1.5, max_iter=5)
print("iterates:", np.round(hist, 2))

**Lesson:** use Newton when you have a good starting guess (from a plot, or a physical approximation) and a derivative. Always check that the answer actually satisfies $f(x) \approx 0$ and is the root you wanted.

### Your turn 2

**Kepler's equation.** A planet on an elliptical orbit with eccentricity $e$ reaches its position at time $t$ described by the "eccentric anomaly" $E$, which satisfies

$$
M = E - e\sin E,
$$

where $M = 2\pi t/T$ grows uniformly with time ($T$ is the orbital period). Given $M$, you must solve for $E$; this is the root-finding problem at the heart of every orbit simulation. For $e = 0.5$ and $M = 1$:

1. Write $f(E)$ and $f'(E)$ and solve with `newton` starting from $E_0 = M$.
2. How many iterations did it take? Check your answer by plugging it back in.
3. Try $e = 0.99$ and $M = 0.1$ with the same starting guess. Does Newton still converge quickly?

In [ ]:
# Your code here

## 5. The secant method

Newton needs $f'(x)$. Sometimes you do not have a formula for it (for example, when $f$ itself is computed by a long simulation). The **secant method** replaces the derivative with a finite difference through the two most recent points, which you know from notebook 07:

$$
f'(x_n) \approx \frac{f(x_n) - f(x_{n-1})}{x_n - x_{n-1}}
\quad\Rightarrow\quad
x_{n+1} = x_n - f(x_n)\,\frac{x_n - x_{n-1}}{f(x_n) - f(x_{n-1})}.
$$

It needs two starting values, and converges with order $\approx 1.618$ (the golden ratio!): slower than Newton but much faster than bisection, and only one new function evaluation per step.

In [ ]:
def secant(f, x0, x1, tol=1e-12, max_iter=50):
    """Secant method. Returns (root, list of iterates)."""
    history = [x0, x1]
    for i in range(max_iter):
        x2 = x1 - f(x1) * (x1 - x0) / (f(x1) - f(x0))
        history.append(x2)
        if abs(x2 - x1) < tol:
            break
        x0, x1 = x1, x2
    return x2, history

x_sec, hist_sec = secant(f_wien, 4.0, 6.0)
print(f"root = {x_sec:.12f} after {len(hist_sec) - 2} iterations")

The line `x0, x1 = x1, x2` updates both variables at once: the new `x0` is the old `x1`, and the new `x1` is `x2`.

## 6. Comparing convergence

Let's plot the error $|x_n - x^*|$ versus iteration number for the three methods. We use `ax.semilogy`, which makes only the $y$-axis logarithmic. On such a plot, linear convergence (constant factor per step) is a straight line, while faster convergence curves downward.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for name, hist in [("bisection", hist_bis), ("Newton", hist_newt), ("secant", hist_sec)]:
    err = np.abs(np.array(hist) - x_true)
    err[err == 0] = 1e-17                  # put exact zeros at the bottom of the plot
    ax.semilogy(err, "o-", ms=4, label=name)
ax.set_xlabel("iteration $n$")
ax.set_ylabel(r"error $|x_n - x^*|$")
ax.set_ylim(1e-17, 1e2)
ax.legend()
plt.show()

| Method | Needs | Convergence | Guaranteed? |
|---|---|---|---|
| Bisection | bracket $[a, b]$ with sign change | linear (factor 2 per step) | yes |
| Newton–Raphson | $x_0$ and $f'(x)$ | quadratic (digits double) | no |
| Secant | $x_0$, $x_1$ | order 1.618 | no |

The best practical algorithms combine them: keep a bracket for safety, but take fast secant-like steps when they behave. That is what **Brent's method** does, and it is the default you should reach for.

## 7. Root finding with SciPy

`scipy.optimize` has well-tested versions of all these methods:

- `optimize.brentq(f, a, b)`: Brent's method. Needs a bracket; guaranteed and fast. **Use this by default** for one equation in one unknown.
- `optimize.newton(f, x0, fprime=...)`: Newton's method if you pass `fprime`, the secant method if you don't.
- `optimize.root_scalar(f, ...)`: one interface to all of the above, with a `method` argument; returns an object with details.

In [ ]:
x_brent = optimize.brentq(f_wien, 1, 10)
x_sp_newton = optimize.newton(f_wien, 5.0, fprime=fprime_wien)
x_sp_secant = optimize.newton(f_wien, 5.0)

print(f"brentq:          {x_brent:.15f}")
print(f"newton:          {x_sp_newton:.15f}")
print(f"newton (secant): {x_sp_secant:.15f}")

In [ ]:
sol = optimize.root_scalar(f_wien, bracket=[1, 10], method="brentq")
print(sol)

`root_scalar` returns a *results object*. You get its parts with a dot: `sol.root` is the answer, `sol.iterations` the number of iterations, and `sol.converged` tells you whether it succeeded. **Always check `converged`** (or the equivalent) in real work.

In [ ]:
print(sol.root, sol.iterations, sol.converged)

### 7.1 Systems of equations: `fsolve` (brief)

Sometimes you have several equations in several unknowns. For example, where does the circle $x^2 + y^2 = 4$ intersect the curve $y = e^x - 1$? Write both as "something = 0" and put them in a function that takes an array `p = [x, y]` and returns an array of the two left-hand sides. `optimize.fsolve` uses a multidimensional version of Newton's method, so it needs a starting guess, here read off a sketch.

In [ ]:
def equations(p):
    x, y = p
    return [x**2 + y**2 - 4,
            y - (np.exp(x) - 1)]

solution = optimize.fsolve(equations, [1.0, 1.0])
print("solution (x, y):", solution)
print("check, should be ~0:", np.array(equations(solution)))

(Notice `x, y = p`: unpacking works for arrays and lists, not just function return values.) The curves also intersect at a second point with negative $x$. A different starting guess would find it.

## 8. Physics: Wien's displacement constant

Back to the start. With the root $x^* \approx 4.965$, Wien's constant is $b = hc/(k_B x^*)$. `scipy.constants` lists the measured value as `const.Wien`.

In [ ]:
x_star = optimize.brentq(f_wien, 1, 10)
b_wien = const.h * const.c / (const.k * x_star)

print(f"x*             = {x_star:.10f}")
print(f"b (computed)   = {b_wien:.10e} m K")
print(f"b (scipy)      = {const.Wien:.10e} m K")

A perfect match. Now we can predict peak wavelengths. The Sun's surface is about 5800 K; a person is about 310 K; the cosmic microwave background is 2.725 K.

In [ ]:
for name, T in [("Sun", 5800), ("human body", 310), ("CMB", 2.725)]:
    lam_max = b_wien / T
    print(f"{name:>10}: T = {T:7.1f} K  ->  lambda_max = {lam_max * 1e6:10.3f} micrometers")

The Sun peaks in visible light (0.5 μm), people glow in the infrared (about 9 μm, which is what thermal cameras detect), and the CMB peaks at about 1 mm, in the microwave band.

## 9. Physics: the finite square well

**Physics recap.** In the *infinite* square well (a "particle in a box") the walls are impenetrable and the energies are $E_n = \dfrac{n^2\pi^2\hbar^2}{2mL^2}$ for a box of width $L$. A real well has finite depth $V_0$: the particle has potential energy $-V_0$ inside the well ($-a < x < a$, width $L = 2a$) and $0$ outside. Bound states have energies between $-V_0$ and $0$. Outside the well the wavefunction is not zero but decays exponentially (tunneling into the walls), so the particle has "more room" and its energies are a bit lower than in the infinite well.

Solving the Schrödinger equation inside and outside and matching $\psi$ and $\psi'$ at the walls (see any modern physics text, e.g. Griffiths §2.6) gives conditions on a dimensionless variable $z$:

$$
z = \frac{a}{\hbar}\sqrt{2m(E + V_0)}, \qquad z_0 = \frac{a}{\hbar}\sqrt{2mV_0}.
$$

Here $E + V_0$ is the kinetic energy inside the well (energy measured from the bottom), and $z_0$ measures how deep and wide the well is. The allowed values of $z$ satisfy

$$
\text{even states:}\quad z\tan z = \sqrt{z_0^2 - z^2}, \qquad\qquad
\text{odd states:}\quad -z\cot z = \sqrt{z_0^2 - z^2},
$$

with $0 < z < z_0$. "Even" states are symmetric, $\psi(-x) = \psi(x)$ (like the ground state), and "odd" ones are antisymmetric. Once we have each root $z_n$, the energy above the well bottom is

$$
E_n + V_0 = \frac{\hbar^2 z_n^2}{2ma^2}.
$$

These are transcendental equations, so we need root finding.

### 9.1 Setting up: an electron in a 1 nm, 10 eV well

In [ ]:
m = const.m_e                    # electron mass (kg)
L = 1e-9                         # well width (m)
a = L / 2                        # half-width (m)
V0 = 10 * const.e                # well depth: 10 eV in joules

z0 = a / const.hbar * np.sqrt(2 * m * V0)
print(f"z0 = {z0:.4f}")

### 9.2 Graphical solution

Plot the left-hand sides $z\tan z$ and $-z\cot z$ and the right-hand side $\sqrt{z_0^2 - z^2}$ (a quarter circle of radius $z_0$) on the same axes. Each crossing is a bound state.

$\tan z$ has vertical asymptotes at $z = \pi/2, 3\pi/2, \ldots$, where it jumps from $+\infty$ to $-\infty$. If we just plot it, Matplotlib draws a nearly vertical line connecting the two sides of each jump. To hide those lines, we replace very large values with `np.nan` ("not a number"), which Matplotlib skips. `np.where(condition, A, B)` picks from `A` where the condition is true and from `B` elsewhere.

In [ ]:
z = np.linspace(0.001, z0, 2000)
rhs = np.sqrt(z0**2 - z**2)
lhs_even = z * np.tan(z)
lhs_odd = -z / np.tan(z)

lhs_even = np.where(np.abs(lhs_even) > 30, np.nan, lhs_even)   # hide jumps at asymptotes
lhs_odd = np.where(np.abs(lhs_odd) > 30, np.nan, lhs_odd)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(z, rhs, "k-", lw=2, label=r"$\sqrt{z_0^2 - z^2}$")
ax.plot(z, lhs_even, "C0", label=r"$z\tan z$ (even)")
ax.plot(z, lhs_odd, "C3", label=r"$-z\cot z$ (odd)")
for k in range(1, 6):
    ax.axvline(k * np.pi / 2, color="gray", ls=":", lw=0.8)
ax.set_ylim(0, 1.2 * z0)
ax.set_xlabel("$z$")
ax.set_ylabel("value")
ax.set_title(f"Finite square well, $z_0$ = {z0:.2f}")
ax.legend(loc="upper right")
plt.show()

Count the crossings: the blue (even) and red (odd) branches alternate, each between neighboring dotted lines at multiples of $\pi/2$. For $z_0 \approx 8.1$ there are 6 crossings, so **6 bound states**. In general a new state appears every time $z_0$ passes a multiple of $\pi/2$, so

$$
N_\text{states} = \left\lfloor \frac{2z_0}{\pi} \right\rfloor + 1 ,
$$

where $\lfloor\cdot\rfloor$ means "round down". There is always at least one bound state, no matter how shallow the well.

### 9.3 Finding all the roots: scan, then refine

A robust general recipe for finding *all* roots in an interval:

1. Evaluate $f$ on a fine grid.
2. Find every place where $f$ changes sign between neighboring grid points. Each is a bracket.
3. Refine each bracket with `brentq`.

But there is a trap. Define $f_\text{even}(z) = z\tan z - \sqrt{z_0^2 - z^2}$. It changes sign at every root **and also at every asymptote** of $\tan z$, where it jumps from $+\infty$ to $-\infty$. Bisection-type methods would happily "converge" to the asymptote.

The fix is to multiply through by $\cos z$ (and $\sin z$ for the odd case) to get functions with no poles:

$$
f_\text{even}(z) = z\sin z - \sqrt{z_0^2 - z^2}\,\cos z, \qquad
f_\text{odd}(z) = z\cos z + \sqrt{z_0^2 - z^2}\,\sin z .
$$

Wherever $\cos z \ne 0$, $f_\text{even} = 0$ is the same equation as before, and at $\cos z = 0$ it equals $z\sin z \ne 0$, so we gain no fake roots. (Same for the odd case with $\sin z$.)

In [ ]:
def f_even(z, z0):
    return z * np.sin(z) - np.sqrt(z0**2 - z**2) * np.cos(z)

def f_odd(z, z0):
    return z * np.cos(z) + np.sqrt(z0**2 - z**2) * np.sin(z)

These functions take an extra parameter `z0`. `brentq` can pass extra arguments to `f` through its `args` keyword: `optimize.brentq(f_even, lo, hi, args=(z0,))`. The trailing comma in `(z0,)` makes it a **tuple** with one element, which is what `args` expects.

Here is the scan-and-refine recipe as a function. The comparison `np.sign(y[:-1]) != np.sign(y[1:])` compares each grid point with its right neighbor, giving a boolean array that is `True` wherever the sign changes, and `np.nonzero(...)[0]` returns the indices where it is `True`.

In [ ]:
def find_all_roots(f, lo, hi, args=(), n_grid=2000):
    """Find all sign-change roots of f(z, *args) in [lo, hi] with brentq."""
    zs = np.linspace(lo, hi, n_grid)
    ys = f(zs, *args)
    changes = np.nonzero(np.sign(ys[:-1]) != np.sign(ys[1:]))[0]
    roots = [optimize.brentq(f, zs[i], zs[i + 1], args=args) for i in changes]
    return np.array(roots)

eps_z = 1e-9      # stay just inside (0, z0)
z_even = find_all_roots(f_even, eps_z, z0 - eps_z, args=(z0,))
z_odd = find_all_roots(f_odd, eps_z, z0 - eps_z, args=(z0,))
print("even roots:", z_even)
print("odd roots: ", z_odd)

(In `f(zs, *args)`, the `*` unpacks the tuple `args` into separate arguments, so `f(zs, *(z0,))` is the same as `f(zs, z0)`.)

Three even and three odd roots, matching the graph. Let's merge and sort them, check them against the *original* equations, and convert to energies. We report energies relative to the top of the well ($E_n$, negative for bound states) and above the bottom ($E_n + V_0$).

In [ ]:
z_roots = np.sort(np.concatenate([z_even, z_odd]))
E_above_bottom = const.hbar**2 * z_roots**2 / (2 * m * a**2) / const.e    # eV
E_n = E_above_bottom - 10.0                                                # eV

n = np.arange(1, len(z_roots) + 1)
E_inf = n**2 * np.pi**2 * const.hbar**2 / (2 * m * L**2) / const.e        # infinite well (eV)

print(" n     z_n     E_n (eV)   E_n+V0 (eV)   infinite well (eV)")
for k in range(len(z_roots)):
    print(f"{n[k]:2d}  {z_roots[k]:7.4f}   {E_n[k]:8.4f}   {E_above_bottom[k]:9.4f}   {E_inf[k]:12.4f}")

Note the units: we divided by `const.e` (the elementary charge in coulombs) to convert joules to electron-volts. Two checks:

- **Count:** $\lfloor 2z_0/\pi\rfloor + 1 = 6$. Good.
- **Physics:** every finite-well level lies below the matching infinite-well level, because the wavefunction leaks into the walls and the particle effectively has a wider box. The difference is small for the deeply bound ground state and large near the top, where the leakage is greatest. In fact the infinite-well $n = 6$ level (13.5 eV) is above the top of the finite well entirely.

Let's draw the energy levels.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
well_x = [-1.0, -0.5, -0.5, 0.5, 0.5, 1.0]
well_V = [0, 0, -10, -10, 0, 0]
ax.plot(well_x, well_V, "k-", lw=2)
for k in range(len(E_n)):
    ax.hlines(E_n[k], -0.5, 0, color="C0")
    ax.hlines(E_inf[k] - 10, 0, 0.5, color="C1", ls="--")
ax.text(-0.25, 1.0, "finite", ha="center", color="C0")
ax.text(0.25, 1.0, "infinite", ha="center", color="C1")
ax.set_xlabel("$x$ (nm)")
ax.set_ylabel("energy (eV)")
ax.set_ylim(-10.5, 4.5)
plt.show()

### 9.4 Number of bound states vs well depth

How does the number of bound states grow as the well gets deeper? Let's repeat the root search for a range of depths and compare with the formula $\lfloor 2z_0/\pi\rfloor + 1$.

In [ ]:
depths_eV = np.linspace(0.05, 20, 200)
counts = []
for V0_eV in depths_eV:
    z0_i = a / const.hbar * np.sqrt(2 * m * V0_eV * const.e)
    n_even = len(find_all_roots(f_even, eps_z, z0_i - eps_z, args=(z0_i,)))
    n_odd = len(find_all_roots(f_odd, eps_z, z0_i - eps_z, args=(z0_i,)))
    counts.append(n_even + n_odd)

In [ ]:
z0_all = a / const.hbar * np.sqrt(2 * m * depths_eV * const.e)
formula = np.floor(2 * z0_all / np.pi) + 1

fig, ax = plt.subplots()
ax.plot(depths_eV, counts, "o", ms=3, label="root finding")
ax.plot(depths_eV, formula, "-", label=r"$\lfloor 2z_0/\pi \rfloor + 1$")
ax.set_xlabel("well depth $V_0$ (eV)")
ax.set_ylabel("number of bound states")
ax.legend()
plt.show()

Because $z_0 \propto \sqrt{V_0}$, new states appear more and more slowly as the well gets deeper. (A small disagreement can occur exactly at a threshold, where a new state is just barely bound with $z \approx z_0$.)

### Your turn 3

A simple model of a neutron in a nucleus: a finite square well of depth $V_0 = 40$ MeV and width $L = 2a = 4$ fm (1 fm = $10^{-15}$ m), with the neutron mass `const.m_n`.

1. Compute $z_0$ and predict the number of bound states.
2. Find all the even and odd roots with `find_all_roots` and convert them to energies in MeV (use `const.e * 1e6` for 1 MeV in joules).
3. Compare with the infinite-well energies for the same width.

In [ ]:
# Your code here

### Your turn 4

For the electron in a 1 nm well, compute the ground-state energy above the bottom of the well, $E_1 + V_0$, for well depths $V_0$ from 0.1 eV to 100 eV (use `np.logspace`). Plot it versus $V_0$ on a log $x$-axis, together with a horizontal line at the infinite-well value $\pi^2\hbar^2/(2mL^2)$. Explain the behavior at both ends in a sentence or two.

In [ ]:
# Your code here

## 10. Minimization (brief)

A closely related problem: find the $x$ that makes $f(x)$ as small as possible. Physical systems settle into **minima of the potential energy**, so this comes up constantly. At a smooth minimum $f'(x) = 0$, so you *could* find the root of the derivative. But SciPy also has direct minimizers that need only $f$.

**Physics recap.** In notebook 07 we met the Lennard-Jones potential between two argon atoms,

$$
V(r) = 4\epsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right],
$$

with $\epsilon = 0.0104$ eV and $\sigma = 0.340$ nm. The **equilibrium separation** $r_0$, where a weakly bound argon pair (a "dimer") sits at rest, is the minimum of $V$. Analytically, $r_0 = 2^{1/6}\sigma$.

`optimize.minimize_scalar(V, bracket=(r1, r2))` searches for a minimum starting from two points around it; `method="bounded"` with `bounds=(lo, hi)` restricts the search to an interval.

In [ ]:
eps_LJ, sig_LJ = 0.0104, 0.340              # eV, nm

def V_LJ(r):
    return 4 * eps_LJ * ((sig_LJ / r)**12 - (sig_LJ / r)**6)

res = optimize.minimize_scalar(V_LJ, bounds=(0.3, 0.6), method="bounded")
print(f"r0 (minimize_scalar) = {res.x:.6f} nm,  V(r0) = {res.fun:.6f} eV")
print(f"r0 (analytic)        = {2**(1/6) * sig_LJ:.6f} nm,  V = {-eps_LJ} eV")

The well depth $V(r_0) = -\epsilon$ is the binding energy of the dimer, about 10 meV, which is why argon is a gas at room temperature ($k_BT \approx 25$ meV).

Notice that the minimizer agrees with the analytic $r_0$ to only about 5 digits. One reason is the default tolerance of the bounded method, $10^{-5}$ in $r$. We can ask for a tighter one with the `options` dictionary:

In [ ]:
for tol in [1e-5, 1e-8, 1e-12]:
    res = optimize.minimize_scalar(V_LJ, bounds=(0.3, 0.6), method="bounded", options={"xatol": tol})
    print(f"xatol = {tol:.0e}:  r0 = {res.x:.15f} nm,  error = {res.x - 2**(1/6) * sig_LJ:.1e} nm")

Tightening the tolerance helps, but only down to an error of about $10^{-10}$ nm (about 9 digits), not the 15–16 digits we got for roots. The deeper reason: near a minimum, $V$ is flat: $V(r_0 + \delta) \approx V(r_0) + \tfrac12 V''\delta^2$, so a *relative* change of $10^{-8}$ in $r$ changes $V$ by only a relative amount of order $(10^{-8})^2 = 10^{-16}$, which is lost to round-off. The minimizer simply cannot tell those points apart. **Minima can be located only to about $\sqrt{\varepsilon}$ relative precision**, while roots can be found to $\varepsilon$. If you need $r_0$ precisely, find the root of the force $F = -dV/dr$ instead.

### Your turn 5

Find $r_0$ a second way: write the exact force $F(r) = -dV/dr$ (or use a central difference from notebook 07), and find its root with `brentq` on the interval $[0.3, 0.6]$ nm. How many digits agree with $2^{1/6}\sigma$ now?

In [ ]:
# Your code here

## Summary

- Write any equation as $f(x) = 0$. **Plot $f$ first** to see how many roots there are and where.
- **Bisection:** needs a bracket with a sign change; always converges; gains one binary digit per step ($n \approx \log_2[(b-a)/\delta]$ iterations).
- **Newton–Raphson:** $x_{n+1} = x_n - f(x_n)/f'(x_n)$; quadratic convergence (digits double each step) from a good starting guess; can jump to another root, blow up when $f' \approx 0$, cycle, or diverge. Always cap the iterations.
- **Secant:** Newton with a finite-difference derivative; order 1.618; needs no $f'$.
- **SciPy:** `optimize.brentq(f, a, b)` (default choice), `optimize.newton(f, x0, fprime=...)`, `optimize.root_scalar(...)` (check `.converged`), `optimize.fsolve(F, x0)` for systems, `optimize.minimize_scalar(f, ...)` for minima.
- **All roots in an interval:** scan a grid for sign changes, then refine each bracket with `brentq`. Watch out for **poles**, which also change sign; rewrite $f$ to remove them.
- Physics: Wien's constant $b = hc/(k_B x^*)$ with $x^* = 4.965$; finite-well bound states from $z\tan z = \sqrt{z_0^2 - z^2}$ and $-z\cot z = \sqrt{z_0^2 - z^2}$, with $\lfloor 2z_0/\pi\rfloor + 1$ states; equilibrium separation as a minimum of $V(r)$.
- New Python: `while` loops, `break`, `list.append`, `enumerate`, simultaneous assignment `a, b = b, a`, `np.where`, `np.nan`, `np.sign`, `np.nonzero`, `args=(...,)` tuples, `*args` unpacking, results objects (`sol.root`).

## Further reading

- M. Newman, *Computational Physics*, Chapter 6.3 (nonlinear equations), free chapters at <https://websites.umich.edu/~mejn/cp/>.
- SciPy documentation: [Optimization and root finding (`scipy.optimize`)](https://docs.scipy.org/doc/scipy/reference/optimize.html).
- D. J. Griffiths, *Introduction to Quantum Mechanics*, Section 2.6, for the finite square well derivation.